In [ ]:
###--- Load libraries and set the location path for analysis data ---###

In [ ]:
import numpy as np
import scanpy as sc
import pandas as pd
import scipy.io
import matplotlib as mpl
import batchglm.api as glm
import diffxpy.api as de
import decoupler as dc

from matplotlib import rcParams
import bbknn
import os
import sys
import scipy
import seaborn as sns
import scipy.io as sio
import scanpy.external as sce
import matplotlib.pyplot as plt
import scipy.sparse as sp
import scrublet as scr

In [ ]:
sc.settings.verbosity = 2  # show logging output
sc.settings.dir = "scRNA_out/"
sc.settings.autosave = True  # save figures, do not show them
sc.settings.figdir = "scRNA_out/figure"
sc.settings.set_figure_params(dpi=200, format="pdf", dpi_save=400)

In [ ]:
###--- Load the raw scRNA-seq data ---###

In [ ]:
# Data Loading: 
file_path = "DGE.mtx"
data = scipy.io.mmread(file_path)
cell_metadata = pd.read_csv("cell_metadata.csv")
all_genes = pd.read_csv("all_genes.csv")

In [ ]:
# Loding information cell_metadata
adata = sc.AnnData(X=data)
adata.X = adata.X.toarray()
adata.obs_names = cell_metadata.index
adata.obs = cell_metadata

In [ ]:
adata.var = all_genes

In [ ]:
adata.var_names = adata.var['gene_name']

In [ ]:
# cell_metadata
donor = []
for i in range(len(adata)):
    text = adata.obs['sample'][i]
    if text.startswith('UT1'):
        donor.append('BC0066'),
    if text.startswith('UT2'):
        donor.append('BC0066')
    if text.startswith('UT3'):
        donor.append('BC0064')
    if text.startswith('CAR1'):
        donor.append('BC0003'),
    if text.startswith('CAR2'):
        donor.append('BC0050')
    if text.startswith('CAR3'):
        donor.append('BC00503')
    if text.startswith('Tr2DG1'):
        donor.append('BC00503'),
    if text.startswith('Tr2DG2'):
        donor.append('BC0050')
    if text.startswith('Tr2DG3'):
        donor.append('BC0003')
    if text.startswith('TrTUN1'):
        donor.append('BC0003'),
    if text.startswith('TrTUN2'):
        donor.append('BC00503')
    if text.startswith('TrTUN3'):
        donor.append('BC00503')

adata.obs['donor'] = donor
adata.obs

In [ ]:
label = []
for i in range(len(adata)):
    text = adata.obs['sample'][i]
    if text.startswith('UT'):
        label.append('UT')
    if text.startswith('CAR'):
        label.append('CAR')
    if text.startswith('TrTUN'):
        label.append('TrTUN')
    if text.startswith('Tr2DG'):
        label.append('Tr2DG')

adata.obs['label'] = label
adata.obs

In [ ]:
###--- Cell Quality Control and Filtering ---###

In [ ]:
# Quality Control (QC): Perform quality control to remove low-quality cells and genes.
# Basic filtering
sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=3)

In [ ]:
#A violin plot of some of the computed quality measures:
adata.var['mt'] = adata.var_names.str.startswith('MT-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)
sc.pl.violin(adata, ['n_genes_by_counts', 'total_counts', 'pct_counts_mt'],
             jitter=0.4, multi_panel=True, save='violin_QC')

In [ ]:
# Remove cells that have too many mitochondrial genes expressed or too many total counts:
adata = adata[adata.obs.n_genes_by_counts < 5000, :]
adata = adata[adata.obs.pct_counts_mt < 15, :]
adata = adata[adata.obs.total_counts < 25000, :]

In [ ]:
adata.layers["counts"] = adata.X.copy()

In [ ]:
# Normalization: Normalize the data to remove the technical variation.
sc.pp.normalize_total(adata, target_sum=1e4)
# Logarithmize the data:
sc.pp.log1p(adata)

In [ ]:
#Freeze the state of the AnnData object
adata.raw = adata
adata.raw.var.index.is_unique

In [ ]:
#Scale each gene to unit variance. 
sc.pp.scale(adata, max_value=10)

In [ ]:
###--- Doublet Filtering ---###

In [ ]:
adata

In [ ]:
adata.X = adata.layers["counts"].copy()
# Normalization: Normalize the data to remove the technical variation.
sc.pp.normalize_total(adata, target_sum=1e4)
# Logarithmize the data:
sc.pp.log1p(adata)

In [ ]:
# doublet calling with Scrublet
adata.X = adata.layers["counts"].copy()
adata_scrub = adata.copy()

In [ ]:
# filtering doublets
scrub = scr.Scrublet(adata_scrub.X, expected_doublet_rate=0.03)
doublet_scores, predicted_doublets = scrub.scrub_doublets()

In [ ]:
# Add the results to your AnnData object
adata.obs['doublet_scores'] = doublet_scores
adata.obs['predicted_doublets'] = predicted_doublets

In [ ]:
sc.pl.scatter(adata, x='n_genes_by_counts', y='doublet_scores', save='scatter_doublet_scores.pdf')

In [ ]:
# High filtering strategies to improve data quality
threshold = 0.25
adata = adata[adata.obs['doublet_scores'] < threshold].copy()

In [ ]:
adata

In [ ]:
del scrub

In [ ]:
###--- Integration with Harmony ---###

In [ ]:
# Highly Variable Gene Selection: Identify highly variable genes to focus on the most informative genes.
sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.2)
sc.pl.highly_variable_genes(adata)

In [ ]:
#Freeze the state of the AnnData object
adata.raw = adata
adata.raw.var.index.is_unique

In [ ]:
#Scale each gene to unit variance. 
sc.pp.scale(adata, max_value=10)

In [ ]:
#Dimensionality Reduction
sc.pp.pca(adata, svd_solver="arpack", use_highly_variable=True,  n_comps=60)
sc.pl.pca_variance_ratio(adata, log=True, n_pcs=60)

In [ ]:
#Computing the neighborhood graph
sc.pp.neighbors(adata, n_neighbors=15, n_pcs = 50)
sc.tl.umap(adata)

In [ ]:
# Clustering the neighborhood graph
sc.tl.leiden(adata, key_added="leiden", resolution=0.8)

In [ ]:
sc.pl.umap(adata, color=['leiden','donor', 'label', 'sample'],  save='_before_harmony')

In [ ]:
#Running Harmony
sce.pp.harmony_integrate(adata, 'donor')

In [ ]:
adata.obsm['X_pca'] = adata.obsm['X_pca_harmony']

In [ ]:
sc.pp.neighbors(adata, n_neighbors=20, n_pcs = 50)
sc.tl.umap(adata)

In [ ]:
sc.tl.leiden(adata, resolution=0.8)

In [ ]:
# to see a specific markers
sc.pl.umap(adata, color=['EPCAM','PTPRC'], save='_epcampos_CD45pos_harmony')

In [ ]:
sc.pl.umap(adata, color=['leiden'], legend_loc='on data',  save='_cluster_after_harmony')

In [ ]:
# Save the state of the AnnData object Counts.
adata = adata.copy()
adata.write("sc_clustering_global_harmony.h5ad")